# 准备数据

In [ ]:
# 加载模块
from datetime import datetime

from tqdm import tqdm
from xtquant import xtdata

from vnpy.trader.database import DB_TZ

from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import HistoryRequest

from vnpy.alpha.data_store import DataStore
from vnpy.alpha.index_manager import IndexManager
from vnpy.alpha import logger

from logging import INFO
from vnpy.trader.setting import SETTINGS
# SETTINGS["log.active"] = True
# SETTINGS["log.level"] = INFO
# SETTINGS["log.file"] = False
# SETTINGS["log.console"] = True

SETTINGS["database.name"] = "postgresql"
SETTINGS["database.host"] = "localhost"
SETTINGS["database.port"] = "5432"
SETTINGS["database.database"] = "vnpy"
SETTINGS["database.user"] = "vnpy"
SETTINGS["database.password"] = "vnpy"
# 配置数据服务
SETTINGS["datafeed.name"] = "xt"
SETTINGS["datafeed.username"] = "client"
SETTINGS["datafeed.password"] = ""

from vnpy.trader.datafeed import get_datafeed

In [ ]:
# 设置下载参数
task_name = "csi300"
index_symbol = "000300.SSE"
xt_index_symbol = "000300.SH"

start_date = "20240101"
end_date = "20260415"

intervals = [
    Interval.DAILY,
]

# 初始化数据层和索引层
data_store = DataStore("./lab", source="xt")
index_manager = IndexManager("./lab")

In [ ]:
# 数据层和索引层已初始化
print(f"数据存储路径: {data_store.daily_path}")
print(f"索引存储路径: {index_manager.index_path}")

In [4]:
# 初始化数据服务（这里配置使用的迅投研）
from vnpy_xt.xt_datafeed import XtDatafeed

datafeed = XtDatafeed()
print(f"datafeed type: {type(datafeed)}")
result = datafeed.init()
print(f"init result: {result}")

datafeed type: <class 'vnpy_xt.xt_datafeed.XtDatafeed'>
init result: True


In [ ]:
# 下载全部A股数据（数据层）
sector_name = '沪深A股'
code_list = xtdata.get_stock_list_in_sector(sector_name)
print(f'{sector_name}一共有{len(code_list)}只品种')

# 创建指数配置（支持多指数）
index_manager.create_index("csi300", "沪深300", "000300.SH")
index_manager.create_index("csi500", "中证500", "000905.SH")
index_manager.create_index("sse50", "上证50", "000016.SH")

In [ ]:
# 查询交易日历
days = xtdata.get_trading_dates(market="SZ", start_time=start_date, end_time=end_date)
print(f"交易日数: {len(days)}")

# 下载每日成分股到索引层
end_datetime = datetime.strptime(end_date, "%Y%m%d").replace(tzinfo=DB_TZ)

for ts in tqdm(days, desc="下载成分股"):
    # 毫秒时间戳转 datetime
    dt = datetime.fromtimestamp(ts / 1000, tz=DB_TZ)
    if dt > end_datetime:
        continue

    date_str = dt.strftime("%Y-%m-%d")
    
    # 沪深300成分股
    csi300_codes = xtdata.get_index_weight('000300.SH', date_str)
    csi300_symbols = [code.replace(".SH", ".SSE").replace(".SZ", ".SZSE") 
                      for code in csi300_codes.keys()]
    index_manager.save_components("csi300", date_str, csi300_symbols)
    
    # 中证500成分股  
    csi500_codes = xtdata.get_index_weight('000905.SH', date_str)
    csi500_symbols = [code.replace(".SH", ".SSE").replace(".SZ", ".SZSE") 
                      for code in csi500_codes.keys()]
    index_manager.save_components("csi500", date_str, csi500_symbols)
    
    # 上证50成分股
    sse50_codes = xtdata.get_index_weight('000016.SH', date_str)
    sse50_symbols = [code.replace(".SH", ".SSE").replace(".SZ", ".SZSE") 
                     for code in sse50_codes.keys()]
    index_manager.save_components("sse50", date_str, sse50_symbols)

print(f"\n成分股下载完成!")
print(f"  沪深300: {len(csi300_symbols)}只")
print(f"  中证500: {len(csi500_symbols)}只") 
print(f"  上证50: {len(sse50_symbols)}只")

In [ ]:
# 加载指数成分股代码（从索引层）
start = datetime.strptime(start_date, "%Y%m%d")
end = datetime.strptime(end_date, "%Y%m%d")

# 获取沪深300成分股
component_symbols = index_manager.get_all_symbols("csi300", start, end)
print(f"沪深300成分股数量: {len(component_symbols)}")

# 获取中证500成分股
csi500_symbols = index_manager.get_all_symbols("csi500", start, end)
print(f"中证500成分股数量: {len(csi500_symbols)}")

# 合并去重（下载全部A股数据只需要下载一次）
all_symbols = list(set(component_symbols + csi500_symbols + code_list))
print(f"需要下载的总股票数: {len(all_symbols)}")

In [ ]:
# 转换时间格式
start = datetime.strptime(start_date, "%Y%m%d")
start.replace(tzinfo=DB_TZ)

end = datetime.strptime(end_date, "%Y%m%d")
end.replace(tzinfo=DB_TZ)

# 下载全部A股K线数据到数据层（只需下载一次）
for vt_symbol in tqdm(all_symbols, desc="下载K线数据"):
    symbol, exchange_str = vt_symbol.split(".")
    
    for interval in intervals:
        req = HistoryRequest(symbol, Exchange(exchange_str), start, end, interval)
        bars = datafeed.query_bar_history(req)

        if bars:
            data_store.save_bars(bars)
        else:
            logger.error(f"{interval.value} {vt_symbol} 数据下载失败")

In [ ]:
# 数据下载完成！
print("=" * 60)
print("数据下载完成")
print("=" * 60)
print(f"\n新架构目录结构:")
print(f"  数据层: {data_store.daily_path}")
print(f"  索引层: {index_manager.index_path}")
print(f"\n支持的指数:")
for idx in index_manager.list_indices():
    info = index_manager.get_index_info(idx)
    print(f"  - {idx}: {info['name'] if info else 'N/A'}")
print(f"\n使用示例:")
print(f"  from vnpy.alpha.lab_v2 import AlphaLabV2")
print(f'  lab = AlphaLabV2("./lab", "my_strategy", "xt", "csi300")')